In [1]:
import json
from crewai import Agent, Task, Crew, Process, LLM

# TUTORING FLOW

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph

In [ ]:
# --- 1. Helper Function: File Reading ---
def read_text_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file {file_path}: {str(e)}"

# --- 2. Konfigurasi LLM ---
llm = LLM(
    model="ollama/llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.1  # Low temp agar output JSON konsisten
)

# Load Knowledge Base
# 1. Panduan Pseudocode (untuk Style Checker)
pseudocode_path = '/home/ilham/Documents/python/crewai-vs-langgraph/doc/Pseudocode dan Golang Dasar.md'
pseudocode_knowledge = read_text_file(pseudocode_path)

# 2. Daftar Miskonsepsi (untuk Logic Checker)
misconceptions_path = '/home/ilham/Documents/python/crewai-vs-langgraph/doc/List of misconceptions.md'
misconceptions_knowledge = read_text_file(misconceptions_path)

# --- 3. Definisi Agen (Sub-Agents & Supervisor) ---

# Sub-Agent 1: Fokus pada gaya penulisan, deklarasi variabel, dan typo
style_checker_agent = Agent(
    role="Style & Syntax Auditor",
    goal="Mendeteksi kesalahan format, deklarasi variabel, dan kepatuhan standar penulisan pseudocode.",
    backstory=(
        "Anda adalah asisten dosen yang sangat teliti terhadap detail penulisan.\n"
        "Anda hanya peduli pada: Kamus variabel, Tipe data, Konsistensi nama variabel (typo), "
        "dan struktur dasar (program/endprogram).\n"
        f"Acuan Standar:\n{pseudocode_knowledge}"
    ),
    llm=llm,
    verbose=True
)

# Sub-Agent 2: Fokus pada alur logika dibandingkan dengan solusi kunci
logic_checker_agent = Agent(
    role="Algorithmic Logic Analyst",
    goal="Membandingkan logika siswa dengan solusi referensi (context solution) untuk menemukan kesalahan alur dan miskonsepsi.",
    backstory=(
        "Anda adalah ahli algoritma.\n"
        "Tugas Anda membandingkan 'pseudocode siswa' dengan 'context solution'.\n"
        "Cek: Kondisi If/Else (terbalik atau tidak), Perulangan, Operasi Matematika, dan Edge Cases.\n"
        "Anda juga harus mengidentifikasi apakah siswa mengalami Miskonsepsi tertentu dari daftar berikut.\n"
        f"Daftar Miskonsepsi:\n{misconceptions_knowledge}\n"
        "Abaikan masalah typo variabel (itu tugas Style Auditor), fokuslah hanya pada LOGIKA dan MISKONSEPSI."
    ),
    llm=llm,
    verbose=True
)

# Supervisor: Menggabungkan laporan, menghitung skor, dan format JSON
scoring_supervisor_agent = Agent(
    role="Scoring Supervisor",
    goal="Mengkonsolidasi laporan dari Sub-agent, menghitung skor akhir berdasarkan rubrik, dan output JSON final.",
    backstory=(
        "Anda adalah Kepala Penilai.\n"
        "Anda menerima laporan dari Style Auditor dan Logic Analyst.\n"
        "Tugas Anda:\n"
        "1. Menghitung pengurangan poin berdasarkan 'general rubrication'.\n"
        "2. Menentukan apakah jawaban 'correct' (benar secara fungsional).\n"
        "3. Merangkum kesalahan (summary & misconceptions).\n"
        "4. WAJIB menghasilkan output dalam format JSON murni."
    ),
    llm=llm,
    verbose=True
)

# --- 4. Definisi Task ---

# Task 1: Cek Style
style_task = Task(
    description=(
        "Analisis `pseudocode` siswa berikut berdasarkan standar.\n"
        "Cari kesalahan: Deklarasi variabel hilang, Typo nama variabel, Format tidak standar.\n"
        "Pseudocode Siswa:\n{pseudocode}"
    ),
    expected_output="Daftar poin kesalahan styling dan sintaksis.",
    agent=style_checker_agent
)

# Task 2: Cek Logika
logic_task = Task(
    description=(
        "Analisis logika `pseudocode` siswa untuk `problem` ini.\n"
        "Bandingkan dengan `context_solution`.\n"
        "Problem: {problem}\n"
        "Context Solution: {context_solution}\n"
        "Pseudocode Siswa: {pseudocode}\n\n"
        "Identifikasi kesalahan logika fatal (mayor) atau minor.\n"
        "Sebutkan jika ada indikasi miskonsepsi (Intentional Bug, While Demon, dll)."
    ),
    expected_output="Daftar poin kesalahan logika, perbandingan dengan solusi, dan deteksi miskonsepsi.",
    agent=logic_checker_agent
)

# Task 3: Scoring & JSON Formatting
scoring_task = Task(
    description=(
        "Berdasarkan laporan Style Auditor dan Logic Analyst, lakukan penilaian final.\n"
        "Gunakan `general_rubrication` untuk menghitung pengurangan poin dari total 100 (atau sesuai rubrik).\n\n"
        "Rubrik: {general_rubrication}\n\n"
        "Hasilkan output HANYA dalam format JSON (tanpa markdown ```json ... ```).\n"
        "Format JSON yang WAJIB diikuti:\n"
        "{\n"
        '  "score": "nilai angka (misal: 85)",\n'
        '  "correct": true/false,\n'
        '  "summary": "Ringkasan singkat penilaian",\n'
        '  "Misconceptions": "Daftar kesalahpahaman utama siswa (ambil dari laporan Logic Analyst)",\n'
        '  "pseudocode": "Salin ulang pseudocode siswa di sini"\n'
        "}"
    ),
    expected_output="Valid JSON String only.",
    agent=scoring_supervisor_agent,
    context=[style_task, logic_task] # Supervisor menerima hasil dari task sebelumnya
)

# --- 5. Definisi Crew ---
grading_crew = Crew(
    agents=[style_checker_agent, logic_checker_agent, scoring_supervisor_agent],
    tasks=[style_task, logic_task, scoring_task],
    process=Process.hierarchical # Sequential agar Supervisor mendapat konteks sub-agent
)

# --- 6. Data Input & Eksekusi ---

# Contoh Data Input
input_data = {
    'problem': "Buatlah algoritma untuk menentukan apakah sebuah bilangan N adalah Ganjil atau Genap. Output 'Ganjil' atau 'Genap'.",
    
    'context_solution': """
        Program GanjilGenap
        kamus
            N : integer
        algoritma
            input(N)
            if (N mod 2 == 0) then
                output("Genap")
            else
                output("Ganjil")
            endif
        endprogram
    """,
    
    'pseudocode': """
        Program CekBilangan
        kamus
            bil : integer
        algoritma
            input(bil)
            if (bil mod 2 = 1) then
                print("Genap")  
            else
                print("Ganjil")
        endprogram
    """,
    
    'general_rubrication': """
        Start Score: 100.
        Pengurangan:
        1. Kesalahan Kritis:
        - Logika inti sepenuhnya salah, menghasilkan output tidak relevan: -50 Poin
        - Program tidak menyelesaikan masalah sama sekali (ada usaha): -90 Poin
        - Lembar jawaban kosong: -50 Poin

        2. Kesalahan Logika Mayor:
        - Gagal menangani salah satu kondisi utama: -40 Poin
        - Perhitungan matematis utama tidak akurat: -40 Poin
        - Variabel tidak tertulis pada kamus: -25 Poin

        3. Kesalahan Logika Minor & Struktur:
        - Gagal menangani kasus khusus (edge case): -25 Poin
        - Tidak menggunakan tipe bentukan (jika diwajibkan): -25 Poin
        - Alur program tidak efisien/berbelit: -15 Poin

        4. Kesalahan Kelengkapan:
        - Tipe data variabel tidak sesuai: -10 Poin
        - Penulisan variabel berbeda (typo) antara kamus & program: -5 Poin
        - Format output tidak sesuai: -5 Poin
    """
}

print("### MEMULAI PROSES PENILAIAN BERTINGKAT ###")
result = grading_crew.kickoff(inputs=input_data)

print("\n\n########################")
print("## HASIL JSON FINAL ##")
print("########################\n")

# Membersihkan output jika LLM menambahkan markdown block secara tidak sengaja
clean_result = str(result).replace("```json", "").replace("```", "").strip()

try:
    # Validasi apakah output benar-benar JSON
    json_output = json.loads(clean_result)
    print(json.dumps(json_output, indent=2))
except json.JSONDecodeError:
    print("Warning: Output Raw (Gagal Parsing JSON Murni):")
    print(clean_result)

### MEMULAI PROSES PENILAIAN BERTINGKAT ###


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Task: Analisis `pseudocode` siswa berikut berdasarkan standar.                                                 │
│  Cari kesalahan: Deklarasi variabel hilang, Typo nama variabel, Format tidak standar.                           │
│  Pseudocode Siswa:                                                                                              │
│                                                                                                                 │
│          Program CekBilangan                                                                                    │
│          kamus                                                                                                  │
│              bil : integer                                                                                      │
│          algoritma                                                                                              │
│              input(bil)                                                                                         │
│              if (bil mod 2 = 1) then                                                                            │
│                  print("Genap")                                                                                 │
│              else                                                                                               │
│                  print("Ganjil")                                                                                │
│          endprogram                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Deklarasi variabel `bil` tidak sesuai dengan standar penulisan pseudocode.                                  │
│  2. Typo nama variabel `bil` pada baris `if (bil mod 2 = 1) then`.                                              │
│  3. Format tidak standar pada baris `print("Genap")` dan `print("Ganjil")`.                                     │
│                                                                                                                 │
│  Pseudocode yang benar:                                                                                         │
│                                                                                                                 │
│          Program CekBilangan                                                                                    │
│          kamus                                                                                                  │
│              bil : integer                                                                                      │
│          algoritma                                                                                              │
│              input(bil)                                                                                         │
│              if (bil mod 2 = 0) then                                                                            │
│                  print("Genap")                                                                                 │
│              else                                                                                               │
│                  print("Ganjil")                                                                                │
│          endprogram                                                                                             │
│                                                                                                                 │
│  Kesalahan yang ditemukan adalah:                                                                               │
│                                                                                                                 │
│  * Deklarasi variabel `bil` tidak sesuai dengan standar penulisan pseudocode.                                   │
│  * Typo nama variabel `bil` pada baris `if (bil mod 2 = 1) then`.                                               │
│  * Format tidak standar pada baris `print("Genap")` dan `print("Ganjil")`.                                      │
│                                                                                                                 │
│  Pseudocode yang benar telah diperbaiki dengan mengganti `mod 2 = 1` menjadi `mod 2 = 0` untuk memenuhi         │
│  kondisi bilangan genap.                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Algorithmic Logic Analyst                                                                               │
│                                                                                                                 │
│  Task: Analisis logika `pseudocode` siswa untuk `problem` ini.                                                  │
│  Bandingkan dengan `context_solution`.                                                                          │
│  Problem: Buatlah algoritma untuk menentukan apakah sebuah bilangan N adalah Ganjil atau Genap. Output          │
│  'Ganjil' atau 'Genap'.                                                                                         │
│  Context Solution:                                                                                              │
│          Program GanjilGenap                                                                                    │
│          kamus                                                                                                  │
│              N : integer                                                                                        │
│          algoritma                                                                                              │
│              input(N)                                                                                           │
│              if (N mod 2 == 0) then                                                                             │
│                  output("Genap")                                                                                │
│              else                                                                                               │
│                  output("Ganjil")                                                                               │
│              endif                                                                                              │
│          endprogram                                                                                             │
│                                                                                                                 │
│  Pseudocode Siswa:                                                                                              │
│          Program CekBilangan                                                                                    │
│          kamus                                                                                                  │
│              bil : integer                                                                                      │
│          algoritma                                                                                              │
│              input(bil)                                                                                         │
│              if (bil mod 2 = 1) then                                                                            │
│                  print("Genap")                                                                                 │
│              else                                                                                               │
│                  print("Ganjil")                                                                                │
│          endprogram                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│  Identifikasi kesalahan logika fatal (mayor) atau minor

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Algorithmic Logic Analyst                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Kesalahan logika fatal pada pseudocode siswa adalah:                                                           │
│                                                                                                                 │
│  1. Kondisi `if (bil mod 2 = 1)` tidak benar karena bilangan ganjil memiliki sisa pembagian 2 yang tidak sama   │
│  dengan nol, tetapi kondisinya salah karena menggunakan operator `=` yang menunjukkan kesamaan, bukan           │
│  kesetaraan.                                                                                                    │
│  2. Kondisi `if (bil mod 2 = 1)` juga mengandung miskonsepsi "Intentional Bug" karena asumsi bahwa sistem       │
│  dapat membuat pilihan berdasarkan keadaan masa depan mesin.                                                    │
│                                                                                                                 │
│  Perbandingan dengan solusi referensi menunjukkan bahwa pseudocode siswa memiliki kesalahan logika fatal dan    │
│  miskonsepsi. Solusi referensi menggunakan kondisi `if (N mod 2 == 0)` yang benar untuk membedakan antara       │
│  bilangan ganjil dan genap.                                                                                     │
│                                                                                                                 │
│  Kesalahan minor pada pseudocode siswa adalah:                                                                  │
│                                                                                                                 │
│  1. Deklarasi variabel `bil` tidak sesuai dengan standar penulisan pseudocode.                                  │
│  2. Typo nama variabel `bil` pada baris `if (bil mod 2 = 1) then`.                                              │
│  3. Format tidak standar pada baris `print("Genap")` dan `print("Ganjil")`.                                     │
│                                                                                                                 │
│  Pseudocode yang benar telah diperbaiki dengan mengganti `mod 2 = 1` menjadi `mod 2 == 0` untuk memenuhi        │
│  kondisi bilangan genap.                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scoring Supervisor                                                                                      │
│                                                                                                                 │
│  Task: Berdasarkan laporan Style Auditor dan Logic Analyst, lakukan penilaian final.                            │
│  Gunakan `general_rubrication` untuk menghitung pengurangan poin dari total 100 (atau sesuai rubrik).           │
│                                                                                                                 │
│  Rubrik:                                                                                                        │
│          Start Score: 100.                                                                                      │
│          Pengurangan:                                                                                           │
│          1. Kesalahan Kritis:                                                                                   │
│          - Logika inti sepenuhnya salah, menghasilkan output tidak relevan: -50 Poin                            │
│          - Program tidak menyelesaikan masalah sama sekali (ada usaha): -90 Poin                                │
│          - Lembar jawaban kosong: -50 Poin                                                                      │
│                                                                                                                 │
│          2. Kesalahan Logika Mayor:                                                                             │
│          - Gagal menangani salah satu kondisi utama: -40 Poin                                                   │
│          - Perhitungan matematis utama tidak akurat: -40 Poin                                                   │
│          - Variabel tidak tertulis pada kamus: -25 Poin                                                         │
│                                                                                                                 │
│          3. Kesalahan Logika Minor & Struktur:                                                                  │
│          - Gagal menangani kasus khusus (edge case): -25 Poin                                                   │
│          - Tidak menggunakan tipe bentukan (jika diwajibkan): -25 Poin                                          │
│          - Alur program tidak efisien/berbelit: -15 Poin                                                        │
│                                                                                                                 │
│          4. Kesalahan Kelengkapan:                                                                              │
│          - Tipe data variabel tidak sesuai: -10 Poin                                                            │
│          - Penulisan variabel berbeda (typo) antara kamus & program: -5 Poin                                    │
│          - Format output tidak sesuai: -5 Poin                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Hasilkan output HANYA dalam format JSON (tanpa markdown ```json ... ```).                                      │
│  Format JSON yang WAJIB diikuti:                                                                                │
│  {                                                                                                              │
│    "score": "nilai angka (misal: 85)",                 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scoring Supervisor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "score": "20",                                                                                               │
│    "correct": false,                                                                                            │
│    "summary": "Pseudocode siswa memiliki kesalahan logika fatal dan miskonsepsi.",                              │
│    "Misconceptions": [                                                                                          │
│      "Intentional Bug: Sistem dapat membuat pilihan berdasarkan keadaan masa depan mesin.",                     │
│      "Kondisi if (bil mod 2 = 1) tidak benar karena menggunakan operator '=' yang menunjukkan kesamaan, bukan   │
│  kesetaraan."                                                                                                   │
│    ],                                                                                                           │
│    "pseudocode": "                                                                                              │
│          Program CekBilangan                                                                                    │
│          kamus                                                                                                  │
│              bil : integer                                                                                      │
│          algoritma                                                                                              │
│              input(bil)                                                                                         │
│              if (bil mod 2 = 0) then                                                                            │
│                  print(\"Genap\")                                                                               │
│              else                                                                                               │
│                  print(\"Ganjil\")                                                                              │
│          endprogram                                                                                             │
│  "                                                                                                              │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



########################
## HASIL JSON FINAL ##
########################

{
  "score": "20",
  "correct": false,
  "summary": "Pseudocode siswa memiliki kesalahan logika fatal dan miskonsepsi.",
  "Misconceptions": [
    "Intentional Bug: Sistem dapat membuat pilihan berdasarkan keadaan masa depan mesin.",
    "Kondisi if (bil mod 2 = 1) tidak benar karena menggunakan operator '=' yang menunjukkan kesamaan, bukan kesetaraan."
  ],
  "pseudocode": "
        Program CekBilangan
        kamus
            bil : integer
        algoritma
            input(bil)
            if (bil mod 2 = 0) then
                print(\"Genap\")
            else
                print(\"Ganjil\")
        endprogram
"
}


input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph